### 從資料萃取樣板字典（data-driven template dictionary）

`template_phrases.csv` 目前是人工憑常識草擬的版本，這份 notebook 改成從資料本身建字典：

1. **步驟一**：從 snippet 資料裡找出「高重複片段」（同一段 `llm_input_text` 出現在多篇不同文章），輸出成 CSV 讓人工標註每筆是 `template`（樣板）還是 `reprint`（轉載/通稿）
2. **步驟二**：讀回人工標註結果
3. **方法 A（LCS）**：把標為 `template` 的片段兩兩比對，抓出連續重複出現的最長共同子字串（word-level），代表真正的樣板句式
4. **方法 B（n-gram 對比）**：對標為 `template` 的片段抓 n-gram，用「轉載片段 + 隨機一般片段」當對照組算判別力，避免抓到 `risk`、`uncertainties` 這類在真新聞裡也很常見的詞
5. 合併兩種方法的候選片語、簡單去重後輸出成候選清單，**不會自動覆寫 `template_phrases.csv`**，仍需人工複核、填 `template_type` 後手動合併進去

只新增這一份獨立的 notebook，不會動到 `1_template_flagging.ipynb` 或任何既有輸出。

#### 1. 路徑與參數設定

In [2]:
import os

year = 2024  # 一次處理一個年份，跟其他 notebook 的慣例一致，要換年份就改這裡重新從頭執行

snippet_csv = (
    r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments'
    rf'\output\llm_ready_整年\{year}\llm_ready_data_{year}_context50_snippetlevel.csv'
)

output_dir = r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output'
os.makedirs(output_dir, exist_ok=True)

# 步驟一輸出：待人工標註的高重複片段清單
label_input_csv = os.path.join(output_dir, f'{year}_dedup_candidates_for_labeling.csv')
# 步驟二讀入：人工標註完、回填 label 欄位後另存的檔案（檔名沿用這個，存在同一個資料夾）
label_done_csv = os.path.join(output_dir, f'{year}_dedup_candidates_labeled.csv')
# 最終輸出：從模板標註中萃取出來的候選片語，人工複核後再手動合併進 template_phrases.csv
candidate_phrases_csv = os.path.join(output_dir, f'{year}_template_phrase_candidates.csv')

TOP_N = 300          # 依「不同文章數」取前 N 名的重複片段做人工標註
MIN_LCS_WORDS = 4    # LCS 法：共同子字串至少要幾個字才算候選片語
MIN_LCS_PAIR_COUNT = 2      # LCS 法：同一句子至少要在幾組片段配對裡出現過
NGRAM_RANGE = (3, 8)        # n-gram 對比法：抓幾個字到幾個字的片語
MIN_NGRAM_DOC_COUNT = 2     # n-gram 對比法：至少要在幾篇「模板」片段裡出現過
MIN_DISCRIMINATIVE_SCORE = 5  # n-gram 對比法：模板組出現比例 / 對照組出現比例 的門檻

print('snippet_csv:', snippet_csv)
print('label_input_csv:', label_input_csv)
print('label_done_csv:', label_done_csv)
print('candidate_phrases_csv:', candidate_phrases_csv)

snippet_csv: C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2024\llm_ready_data_2024_context50_snippetlevel.csv
label_input_csv: C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\2024_dedup_candidates_for_labeling.csv
label_done_csv: C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\2024_dedup_candidates_labeled.csv
candidate_phrases_csv: C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\2024_template_phrase_candidates.csv


#### 2. 步驟一：找出高重複片段，輸出人工標註清單

沿用 `1_template_flagging.ipynb` 的頻率法邏輯：對 `llm_input_text` groupby，算「不同文章數」。這裡不套用任何門檻篩掉，直接取「不同文章數」最高的前 `TOP_N` 筆交給人工判斷，因為接下來要人工分辨的正是「這是樣板還是轉載」，不能先用頻率門檻預設答案。

In [3]:
import pandas as pd

print(f"讀取 snippet 檔案: {snippet_csv}")
df_snippet = pd.read_csv(snippet_csv, usecols=['llm_input_text', 'file_name'])
print(f"總列數: {len(df_snippet):,}")

text_stats = df_snippet.groupby('llm_input_text').agg(
    出現次數=('file_name', 'size'),
    不同文章數=('file_name', 'nunique'),
).reset_index()

dup_texts = text_stats[text_stats['不同文章數'] > 1].sort_values('不同文章數', ascending=False)
print(f"重複片段（不同文章數 > 1）共 {len(dup_texts):,} 筆，取前 {TOP_N} 筆做人工標註")

label_batch = dup_texts.head(TOP_N).copy()
label_batch['label'] = ''  # 人工填: template（樣板） 或 reprint（轉載）
label_batch.to_csv(label_input_csv, index=False, encoding='utf-8-sig')

print(f"\n已輸出待標註清單: {label_input_csv}")
print("請打開這個檔案，在 label 欄位填入 template 或 reprint，另存為:")
print(f"  {label_done_csv}")
label_batch.head(10)

讀取 snippet 檔案: C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2024\llm_ready_data_2024_context50_snippetlevel.csv
總列數: 2,207,756
重複片段（不同文章數 > 1）共 298,911 筆，取前 300 筆做人工標註

已輸出待標註清單: C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\2024_dedup_candidates_for_labeling.csv
請打開這個檔案，在 label 欄位填入 template 或 reprint，另存為:
  C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\2024_dedup_candidates_labeled.csv


,llm_input_text,出現次數,不同文章數,label
394767,information on this web site without obligatio...,25001,25001,
394766,information on this web site without obligatio...,6890,6890,
11853,a security this opinion and reports made by fi...,6317,6317,
882631,updates to falcon before rolling them out to c...,590,590,
496637,negative impact on the company s business and ...,544,544,
268618,executing against that strategy and we re seei...,540,540,
898153,was executing against that strategy and we re ...,540,540,
458069,macrogenics inc that seeks to recover losses o...,522,522,
737690,structured finance oct criteria structured fin...,478,478,
700268,seeks to recover losses of shareholders who we...,467,467,


#### 3. 步驟二：讀回人工標註結果（跨年度合併）

讀回 `YEARS` 清單裡每個年份的 `{year}_dedup_candidates_labeled.csv`，把 `template` / `reprint` 片段跨年度合併成一份清單，接下來的 LCS 兩兩比對、n-gram 對比都在這份合併後的清單上做，而不是只用單一年份。

In [6]:
YEARS = [2000, 2006, 2012, 2018, 2024]  # 目前已經標註完成的年份，之後有新年份直接加進這個清單重跑即可

labeled_frames = []
for y in YEARS:
    label_done_csv_y = os.path.join(output_dir, f'{y}_dedup_candidates_labeled.csv')
    if not os.path.exists(label_done_csv_y):
        print(f"[跳過] {y} 年：找不到 {os.path.basename(label_done_csv_y)}")
        continue
    df_y = pd.read_csv(label_done_csv_y, encoding='utf-8-sig')
    df_y['label'] = df_y['label'].astype(str).str.strip().str.lower()
    df_y['year'] = y
    # 只取比對用得到的欄位；不同年份檔案的其他欄位名稱不一致（中英文都有），不影響這裡
    labeled_frames.append(df_y[['year', 'llm_input_text', 'label']])

if not labeled_frames:
    raise FileNotFoundError(f"YEARS 清單 {YEARS} 裡一個標註檔案都找不到，請確認 output 資料夾。")

labeled_all = pd.concat(labeled_frames, ignore_index=True)

template_texts = labeled_all.loc[labeled_all['label'] == 'template', 'llm_input_text'].dropna().tolist()
reprint_texts = labeled_all.loc[labeled_all['label'] == 'reprint', 'llm_input_text'].dropna().tolist()

print(f"跨 {len(labeled_frames)} 個年份合併，各年份 label 分布：")
print(labeled_all.groupby('year')['label'].value_counts().unstack(fill_value=0))
print(f"\n合併後 模板 (template): {len(template_texts):,} 筆")
print(f"合併後 轉載 (reprint): {len(reprint_texts):,} 筆")

if len(template_texts) < 2:
    raise ValueError("標為 template 的片段少於 2 筆，沒辦法做兩兩比對，請先確認標註檔案。")

n_pairs_estimate = len(template_texts) * (len(template_texts) - 1) // 2
print(f"\n下一步 LCS 兩兩比對組數: {n_pairs_estimate:,}"
      f"（實測約 0.1 毫秒/組，預估總耗時約 {n_pairs_estimate * 0.0001 / 60:.1f} 分鐘）")

# 從這步開始，後面每個方法（LCS / n-gram / 合併去重）算完，就把結果新增成這個活頁簿的一個分頁，
# 不再各自輸出成獨立的 CSV，方便在同一個檔案裡對照不同方法的結果
output_workbook_xlsx = os.path.join(output_dir, 'all_years_template_phrase_candidates.xlsx')
with pd.ExcelWriter(output_workbook_xlsx, engine='openpyxl', mode='w') as writer:
    labeled_all.to_excel(writer, sheet_name='labeled_all', index=False)
print(f"\n已建立候選片段總表: {output_workbook_xlsx}（分頁: labeled_all，共 {len(labeled_all):,} 筆）")

跨 5 個年份合併，各年份 label 分布：
label  reprint  template
year                    
2000         0       300
2006         5       295
2012         0       300
2018         0       300
2024        52       248

合併後 模板 (template): 1,443 筆
合併後 轉載 (reprint): 57 筆

下一步 LCS 兩兩比對組數: 1,040,403（實測約 0.1 毫秒/組，預估總耗時約 1.7 分鐘）

已建立候選片段總表: C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\all_years_template_phrase_candidates.xlsx（分頁: labeled_all，共 1,500 筆）


#### 4. 方法 A：模板片段兩兩比對，抓最長共同子字串（LCS, word-level，跨年度合併後的清單）

同一句樣板文字（例如免責聲明）常常被嵌在不同文章、不同上下文的 window 裡，逐字看每個片段並不會完全一樣，但中間一定有一段連續文字完全相同。用 `difflib.SequenceMatcher` 對每兩個 `template` 片段做比對，抓出長度 `>= MIN_LCS_WORDS` 的連續共同子字串，出現在越多組配對裡的子句，越可能是真正的樣板核心句式。

現在 `template_texts` 是 5 個年份合併後的清單（約 1,400 多筆），跨年度比對還有一個額外的好處：同一句樣板套語如果橫跨多個年份都出現（例如 forward-looking statements 的標準警語），比只看單一年份更能確認它是長期穩定的樣板，而不是某一年偶然的巧合用字。

複雜度是 `n*(n-1)/2` 組兩兩配對；實測 `difflib.SequenceMatcher` 在這批文字上大約 0.1 毫秒/組，1,400 多筆全部兩兩比對大約 1~2 分鐘可以跑完，不需要額外做分桶優化。

In [7]:
import itertools
import time
from collections import Counter
from difflib import SequenceMatcher


def extract_common_phrases(text_a, text_b, min_words=MIN_LCS_WORDS):
    """回傳兩段文字之間所有長度 >= min_words 的連續共同子字串（word-level，不重疊）"""
    a, b = text_a.split(), text_b.split()
    sm = SequenceMatcher(None, a, b, autojunk=False)
    phrases = []
    for block in sm.get_matching_blocks():
        if block.size >= min_words:
            phrases.append(' '.join(a[block.a: block.a + block.size]))
    return phrases


lcs_counter = Counter()
n_pairs = len(template_texts) * (len(template_texts) - 1) // 2
print(f"開始兩兩比對 {len(template_texts):,} 筆模板片段（共 {n_pairs:,} 組配對）...")

t0 = time.time()
for i, (text_a, text_b) in enumerate(itertools.combinations(template_texts, 2)):
    for phrase in extract_common_phrases(text_a, text_b):
        lcs_counter[phrase] += 1
    if (i + 1) % 200_000 == 0:
        elapsed = time.time() - t0
        print(f"  已比對 {i + 1:,} / {n_pairs:,} 組（{elapsed:.0f} 秒，"
              f"預估剩餘 {elapsed / (i + 1) * (n_pairs - i - 1):.0f} 秒）")

print(f"兩兩比對完成，耗時 {time.time() - t0:.1f} 秒")

lcs_df = pd.DataFrame(lcs_counter.items(), columns=['phrase', 'pair_count'])
lcs_df = lcs_df[lcs_df['pair_count'] >= MIN_LCS_PAIR_COUNT].sort_values('pair_count', ascending=False)

with pd.ExcelWriter(output_workbook_xlsx, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    lcs_df.to_excel(writer, sheet_name='lcs_candidates', index=False)

print(f"LCS 法找到 {len(lcs_df):,} 種共同子句（pair_count >= {MIN_LCS_PAIR_COUNT}）")
print(f"已新增分頁: lcs_candidates -> {output_workbook_xlsx}")
lcs_df.head(20)

開始兩兩比對 1,443 筆模板片段（共 1,040,403 組配對）...
  已比對 200,000 / 1,040,403 組（22 秒，預估剩餘 91 秒）
  已比對 400,000 / 1,040,403 組（43 秒，預估剩餘 69 秒）
  已比對 600,000 / 1,040,403 組（64 秒，預估剩餘 47 秒）
  已比對 800,000 / 1,040,403 組（85 秒，預估剩餘 25 秒）
  已比對 1,000,000 / 1,040,403 組（105 秒，預估剩餘 4 秒）
兩兩比對完成，耗時 109.2 秒
LCS 法找到 6,291 種共同子句（pair_count >= 2）
已新增分頁: lcs_candidates -> C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\all_years_template_phrase_candidates.xlsx


,phrase,pair_count
6,differ materially from those,30701
81,the forward looking statements,27940
21,such forward looking statements,25037
372,these forward looking statements,22095
142,to differ materially from,22053
494,with the securities and exchange commission,20136
373,forward looking statements are,18471
84,in the forward looking statements,14583
946,to differ materially from those,14286
5,in the company s,13207


#### 5. 方法 B：n-gram 文件頻率法，跟非模板語料對比（對照組跨五個年份抽樣）

單靠 LCS 容易受少數幾篇片段的巧合共同詞干擾，這裡換一個角度：對每篇 `template` 片段各自取一組 n-gram（`NGRAM_RANGE` 個字），統計每個 n-gram 出現在幾篇不同的模板片段裡（document frequency）。同一組 n-gram 也在對照組（`reprint` 片段 + 五個年份各自從 snippet 原始檔隨機抽樣的一般片段）算一次 document frequency，取兩者比例當「判別力」分數——分數高代表這個片語在模板裡很常見、但在一般新聞裡很少見，才是真正該收進字典的樣板詞。

對照組改成每個年份都各自抽樣（`NGRAM_BASELINE_PER_YEAR` 筆/年），而不是只抽單一年份，避免對照組被特定年份的用字習慣或事件類型主導。這一步會重新讀取五個年份的 snippet 原始檔（只取 `llm_input_text` 欄位、去重後抽樣），讀完立即釋放記憶體，不需要事先跑過步驟一。

In [8]:
import gc
from collections import Counter

NGRAM_BASELINE_PER_YEAR = 2000  # 每個年份從原始 snippet 檔案抽樣幾筆當對照組的一部分


def load_baseline_sample_for_year(year, n_sample):
    """讀取該年份 snippet 原始檔的 llm_input_text，去重後隨機抽樣 n_sample 筆；讀完立即釋放記憶體"""
    snippet_csv_y = (
        r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments'
        rf'\output\llm_ready_整年\{year}\llm_ready_data_{year}_context50_snippetlevel.csv'
    )
    if not os.path.exists(snippet_csv_y):
        print(f"  [跳過] {year} 年找不到 snippet 檔案: {snippet_csv_y}")
        return []
    df_y = pd.read_csv(snippet_csv_y, usecols=['llm_input_text'])
    uniq_y = df_y['llm_input_text'].drop_duplicates()
    sample_y = uniq_y.sample(n=min(n_sample, len(uniq_y)), random_state=42).tolist()
    del df_y, uniq_y
    gc.collect()
    return sample_y


def to_ngrams(text, n_range=NGRAM_RANGE):
    """回傳文字裡所有 n_range 範圍內的 n-gram 集合（同一篇文字裡重複出現只算一次）"""
    words = text.split()
    grams = set()
    for n in range(n_range[0], n_range[1] + 1):
        for i in range(len(words) - n + 1):
            grams.add(' '.join(words[i:i + n]))
    return grams


# 模板組的 n-gram 文件頻率
template_doc_freq = Counter()
for text in template_texts:
    template_doc_freq.update(to_ngrams(text))

# 對照組：轉載片段 + 五個年份各自隨機抽樣的一般片段
print(f"從 {len(YEARS)} 個年份的 snippet 原始檔各抽 {NGRAM_BASELINE_PER_YEAR:,} 筆當對照組...")
baseline_sample = []
for y in YEARS:
    sample_y = load_baseline_sample_for_year(y, NGRAM_BASELINE_PER_YEAR)
    print(f"  {y} 年抽到 {len(sample_y):,} 筆")
    baseline_sample.extend(sample_y)

baseline_texts = reprint_texts + baseline_sample

baseline_doc_freq = Counter()
for text in baseline_texts:
    baseline_doc_freq.update(to_ngrams(text))

n_template_docs = len(template_texts)
n_baseline_docs = len(baseline_texts)

rows = []
for phrase, t_count in template_doc_freq.items():
    if t_count < MIN_NGRAM_DOC_COUNT:
        continue
    b_count = baseline_doc_freq.get(phrase, 0)
    template_ratio = t_count / n_template_docs
    baseline_ratio = b_count / n_baseline_docs
    # 加平滑項避免對照組出現次數為 0 時分數爆炸
    discriminative_score = template_ratio / (baseline_ratio + 0.01)
    rows.append({
        'phrase': phrase,
        'word_count': len(phrase.split()),
        'template_doc_count': t_count,
        'template_doc_ratio': round(template_ratio, 4),
        'baseline_doc_count': b_count,
        'baseline_doc_ratio': round(baseline_ratio, 4),
        'discriminative_score': round(discriminative_score, 2),
    })

ngram_df = pd.DataFrame(rows).sort_values('discriminative_score', ascending=False)

with pd.ExcelWriter(output_workbook_xlsx, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    ngram_df.to_excel(writer, sheet_name='ngram_candidates', index=False)

print(f"\nn-gram 候選（模板組至少出現 {MIN_NGRAM_DOC_COUNT} 次）共 {len(ngram_df):,} 筆，對照組樣本數: {n_baseline_docs:,}")
print(f"已新增分頁: ngram_candidates -> {output_workbook_xlsx}")
ngram_df.head(20)

從 5 個年份的 snippet 原始檔各抽 2,000 筆當對照組...
  2000 年抽到 2,000 筆
  2006 年抽到 2,000 筆
  2012 年抽到 2,000 筆
  2018 年抽到 2,000 筆
  2024 年抽到 2,000 筆

n-gram 候選（模板組至少出現 2 次）共 164,551 筆，對照組樣本數: 10,057
已新增分頁: ngram_candidates -> C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\all_years_template_phrase_candidates.xlsx


,phrase,word_count,template_doc_count,template_doc_ratio,baseline_doc_count,baseline_doc_ratio,discriminative_score
4180,are not guarantees,3,133,0.0922,174,0.0173,3.38
4234,statements are not guarantees,4,115,0.0797,138,0.0137,3.36
4069,statements are not guarantees of future,6,110,0.0762,128,0.0127,3.35
4070,statements are not,3,119,0.0825,152,0.0151,3.28
4200,guarantees of future,3,125,0.0866,167,0.0166,3.26
4430,are not guarantees of future,5,121,0.0839,159,0.0158,3.25
4021,not guarantees of future,4,121,0.0839,160,0.0159,3.24
4011,statements are not guarantees of,5,110,0.0762,136,0.0135,3.24
4081,are not guarantees of,4,125,0.0866,171,0.0170,3.21
4488,not guarantees of,3,125,0.0866,172,0.0171,3.20


#### 6. 合併兩種方法、去重，新增最後一個分頁

`discriminative_score >= MIN_DISCRIMINATIVE_SCORE` 篩出 n-gram 法的候選，跟 LCS 法的候選取聯集。因為 n-gram 法本身會把同一句話拆成長度不一、互相包含的好幾個片語（例如 "forward looking statements" 和 "forward looking statements are"都會出現），這裡用「短的若是某個保留下來的長片語的子字串，就丟掉短的」做簡單去重，盡量只留下最長、最完整的版本。

結果新增成 `all_years_template_phrase_candidates.xlsx` 的第四個分頁 `merged_candidates`，`template_type` 欄位留空，需要人工複核、判斷是 disclaimer/rating/copyright/website 哪一類（沿用 `template_phrases.csv` 現有分類）之後，再手動合併進去。整個活頁簿到這裡總共有 4 個分頁：`labeled_all`（步驟二的合併標註總表）、`lcs_candidates`、`ngram_candidates`、`merged_candidates`。

In [9]:
def dedupe_by_containment(phrases_sorted_by_len_desc):
    """依序保留，丟掉已經是某個保留片語子字串的片語"""
    kept = []
    for phrase in phrases_sorted_by_len_desc:
        if not any(phrase in longer for longer in kept):
            kept.append(phrase)
    return kept


lcs_candidates = set(lcs_df['phrase'])
ngram_candidates = set(ngram_df[ngram_df['discriminative_score'] >= MIN_DISCRIMINATIVE_SCORE]['phrase'])

merged_candidates = sorted(lcs_candidates | ngram_candidates, key=len, reverse=True)
final_candidates = dedupe_by_containment(merged_candidates)


def source_label(phrase):
    tags = []
    if phrase in lcs_candidates:
        tags.append('lcs')
    if phrase in ngram_candidates:
        tags.append('ngram')
    return '+'.join(tags)


candidate_df = pd.DataFrame({'phrase': final_candidates})
candidate_df['source'] = candidate_df['phrase'].apply(source_label)
candidate_df['template_type'] = ''  # 人工複核時填入 disclaimer / rating / copyright / website

with pd.ExcelWriter(output_workbook_xlsx, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    candidate_df.to_excel(writer, sheet_name='merged_candidates', index=False)

print(f"合併去重後候選片語數: {len(candidate_df):,}（LCS: {len(lcs_candidates)}，n-gram: {len(ngram_candidates)}）")
print(f"已新增分頁: merged_candidates -> {output_workbook_xlsx}")
print("\n請人工複核這份候選清單、填入 template_type，確認後再手動合併進 template_phrases.csv")
candidate_df.head(30)

合併去重後候選片語數: 1,842（LCS: 6291，n-gram: 0）
已新增分頁: merged_candidates -> C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\all_years_template_phrase_candidates.xlsx

請人工複核這份候選清單、填入 template_type，確認後再手動合併進 template_phrases.csv


,phrase,source,template_type
0,the impact of pharmaceutical industry regulati...,lcs,
1,trend potential opportunity pipeline believe c...,lcs,
2,results discussed in this release and in the f...,lcs,
3,in the united states and internationally globa...,lcs,
4,and involve known and unknown risks uncertaint...,lcs,
5,may cause our actual results levels of activit...,lcs,
6,from those projected factors that could cause ...,lcs,
7,information on this web site without obligatio...,lcs,
8,information on this web site without obligatio...,lcs,
9,the meaning of federal securities regulations ...,lcs,
